In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

## Settings

In [ ]:
# Read .env file
load_dotenv(
    dotenv_path=Path().resolve().parent.parent / ".env",
    override=False,
)

# Set OpenAI API key
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Set Anthropic API key
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

In [ ]:
# Batch input directory
ORIGINAL_TIME_TAG = "20260227220015"
ORIGINAL_BATCH_INPUT_DIR = Path("./batch_input") / f"batch_input_{ORIGINAL_TIME_TAG}"
if not ORIGINAL_BATCH_INPUT_DIR.exists():
    raise FileNotFoundError(f"Batch input directory does not exist: {ORIGINAL_BATCH_INPUT_DIR}")

# Batch output directory
ORIGINAL_BATCH_OUTPUT_DIR = Path("./batch_output") / f"batch_output_{ORIGINAL_TIME_TAG}"
if not ORIGINAL_BATCH_OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Batch output directory does not exist: {ORIGINAL_BATCH_OUTPUT_DIR}")

In [ ]:
# Set directories
DATASET_DIR = Path("./dataset_processed")
SYSTEM_PROMPT_DIR = Path("./system_prompt")
USER_PROMPT_DIR = Path("./user_prompt")

# Create directory for batch jsonl files
TIME_TAG = ORIGINAL_TIME_TAG + "A"  # pd.Timestamp.now().strftime("%Y%m%d%H%M%S")
BATCH_INPUT_DIR = Path("./batch_input") / f"batch_input_{TIME_TAG}"
BATCH_INPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Model lists
OPENAI_NON_REASONING_MODELS = [
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
]
OPENAI_REASONING_MODELS = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
]
OPENAI_MODELS = OPENAI_NON_REASONING_MODELS + OPENAI_REASONING_MODELS

ANTHROPIC_MODELS = [
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]

## Batch processing

In [ ]:
# Read original batch output logs
original_batch_output_logs_df = pd.read_csv(ORIGINAL_BATCH_OUTPUT_DIR / "batch_output_logs.csv")
display(original_batch_output_logs_df)

In [ ]:
# Batch processing
failed_filepath = Path("./results") / f"results_{ORIGINAL_TIME_TAG}_failed.csv"
df_failed = pd.read_csv(failed_filepath)

logs = []
for i, original_batch_output_logs_row in original_batch_output_logs_df.iterrows():
    original_dataset_name = original_batch_output_logs_row["dataset_name"]
    original_model = original_batch_output_logs_row["model"]
    original_max_output_tokens = original_batch_output_logs_row["max_output_tokens"]
    original_reasoning_effort = original_batch_output_logs_row["reasoning_effort"]
    original_temperature = original_batch_output_logs_row["temperature"]
    original_repeat = original_batch_output_logs_row["repeat"]
    original_forward_or_reverse = original_batch_output_logs_row["forward_or_reverse"]
    original_batch_id = original_batch_output_logs_row["batch_id"]
    original_n_samples_input = original_batch_output_logs_row["n_samples_input"]

    print("-" * 100)
    print(
        f"dataset: {original_dataset_name:<15}| model: {original_model:<15}| repeat: {original_repeat}, reverse mode: {original_forward_or_reverse}"
    )

    if original_model in OPENAI_MODELS:
        df_failed_extracted = df_failed.loc[
            (df_failed["dataset_name"] == original_dataset_name)
            & (df_failed["model"] == original_model)
            & (df_failed["repeat"] == original_repeat)
            & (df_failed["forward_or_reverse"] == original_forward_or_reverse)
        ]

        if len(df_failed_extracted) > 0:
            # Read original batch input file
            if original_forward_or_reverse == "forward":
                original_batch_input_filepath = (
                    ORIGINAL_BATCH_INPUT_DIR / f"batch_input_{original_dataset_name}_{original_model}.jsonl"
                )
            elif original_forward_or_reverse == "reverse":
                original_batch_input_filepath = (
                    ORIGINAL_BATCH_INPUT_DIR / f"batch_input_{original_dataset_name}_{original_model}_reverse.jsonl"
                )
            else:
                raise ValueError(f"Unsupported forward_or_reverse value: {original_forward_or_reverse}")
            with open(original_batch_input_filepath, "rb") as f:
                original_lines_input = f.readlines()
            if len(original_lines_input) != original_n_samples_input:
                raise ValueError("Mismatch between original input lines and original n_samples_input.")
            print(f"All: {original_n_samples_input} rows")

            # Filter batch input lines to only include error samples
            requests = []
            for line in original_lines_input:
                original_input_obj = json.loads(line)
                if original_input_obj["custom_id"] in df_failed_extracted["custom_id"].values:
                    requests.append(line)

            print(f"Filtered: {len(requests)} rows")
            if len(requests) != len(df_failed_extracted):
                raise ValueError("Mismatch between filtered input lines and error samples.")

            # Save filtered batch input file
            if original_forward_or_reverse == "forward":
                batch_input_filepath = (
                    BATCH_INPUT_DIR / f"batch_input_{original_dataset_name}_{original_model}_rep{original_repeat}.jsonl"
                )
            elif original_forward_or_reverse == "reverse":
                batch_input_filepath = (
                    BATCH_INPUT_DIR
                    / f"batch_input_{original_dataset_name}_{original_model}_rep{original_repeat}_reverse.jsonl"
                )
            else:
                raise ValueError(f"Unsupported forward_or_reverse value: {original_forward_or_reverse}")
            with open(batch_input_filepath, "wb") as f:
                for request in requests:
                    f.write(request)

            # Read batch input file
            with open(batch_input_filepath, "rb") as f:
                lines = f.readlines()
                n_samples_input = len(lines)
            print(f"{n_samples_input} rows")

            # Batch processing
            client_openai = OpenAI(api_key=OPENAI_API_KEY)
            batch_input_file = client_openai.files.create(file=open(batch_input_filepath, "rb"), purpose="batch")
            batch_input_file_id = batch_input_file.id
            batch = client_openai.batches.create(
                input_file_id=batch_input_file.id,
                endpoint="/v1/responses",
                completion_window="24h",
            )
            batch_id = batch.id
            print(f"Batch ID: {batch_id}")

            log_dict = {
                "dataset_name": original_dataset_name,
                "model": original_model,
                "max_output_tokens": original_max_output_tokens,
                "reasoning_effort": original_reasoning_effort,
                "temperature": original_temperature,
                "n_samples_input": n_samples_input,
                "repeat": original_repeat,
                "forward_or_reverse": original_forward_or_reverse,
                "batch_input_file_id": batch_input_file_id,
                "batch_id": batch_id,
            }

            logs.append(log_dict)
    else:
        raise ValueError(f"Unsupported model: {original_model}")

# Save logs
logs_df = pd.DataFrame(logs)
logs_df.to_csv(BATCH_INPUT_DIR / "batch_input_logs.csv", index=False)